# Variant annotation

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import myvariant
import sys
sys.path.append("code")
from src.pyensembl import * 
from src.utils import *
from src.variant_annotation import *

In [25]:
mane = get_mane_transcripts()
db = get_db()

19284 MANE transcripts found.


## `myvariant`

In [2]:
# See here for an explanation of each field: 
# https://useast.ensembl.org/info/genome/variation/prediction/protein_function.html
mv = get_mv_db()
fields = mv.get_fields()
tx_fields = find_field('consequence')
tx_fields

['cadd.consequence']

In [101]:
pd.set_option('display.max_columns', None)

variants_df = pd.DataFrame()
for transcript_id in mane['TranscriptId'].tolist()[:3]:
    tx = db.transcript_by_id(transcript_id)
    ranges =[f"chr{tx.contig}:{range[0]}-{range[1]}" for range in tx.coding_sequence_position_ranges]
    variant_df_i = mv.query(q=ranges,
        # f"dbnsfp.ensembl.transcriptid:{transcript_id}",
                        #  scopes=['clinvar'],
                         fields='all',  # Specify additional fields as needed
                         assembly='hg38',
                         as_dataframe=True)
    variant_df_i.insert(0, 'transcript_id', transcript_id)
    variants_df = pd.concat([variants_df, variant_df_i])
    print(f"Found {len(variant_df_i)} variants in {transcript_id}")
print(variants_df.shape)

Found 10 variants in ENST00000373020
Found 10 variants in ENST00000373031
Found 10 variants in ENST00000371588
(30, 647)


In [108]:
# drop cols with all nan
# variants_df.dropna(axis=1, how='all').head(50)
# view only clinvar columns
# clinvar_cols = [col for col in variants_df.columns if 'clinvar' in col]
# variants_df[clinvar_cols]
variants_df.columns.tolist()
variants_df.head(10)

# snpeff.ann.putative_impact

KeyError: 'cadd.consequence'

## EMBL-EBI Proteins API

In [19]:
pd.set_option('display.max_columns', None)
embl_res = embl_get_variants('O43657')
embl_res.loc[embl_res['clinicalSignificances'].notna(),]

,accession,entryName,proteinName,geneName,organismName,proteinExistence,sequence,sequenceChecksum,sequenceVersion,taxid,type,alternativeSequence,begin,end,xrefs,cytogeneticBand,genomicLocation,locations,codon,consequenceType,wildType,mutatedType,predictions,somaticStatus,sourceType,descriptions,clinicalSignificances,ftId,evidences,populationFrequencies
15,O43657,TSN6_HUMAN,Tetraspanin-6,TSPAN6,Homo sapiens,Evidence at protein level,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...,9304343482296458210,1,9606,VARIANT,I,10,10,"[{'name': 'ExAC', 'id': 'rs752076056', 'url': ...",Xq22.1,[NC_000023.11:g.100636666G>A],"[{'loc': 'p.Thr10Ile', 'seqId': 'ENST000003730...",ACT/ATT,missense,T,I,"[{'predictionValType': 'possibly damaging', 'p...",0,large_scale_study,NaN,"[{'type': 'Variant of uncertain significance',...",NaN,NaN,NaN
58,O43657,TSN6_HUMAN,Tetraspanin-6,TSPAN6,Homo sapiens,Evidence at protein level,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...,9304343482296458210,1,9606,VARIANT,L,63,63,"[{'name': 'ExAC', 'id': 'rs767429263', 'url': ...",Xq22.1,[NC_000023.11:g.100635647C>G],"[{'loc': 'p.Val63Leu', 'seqId': 'ENST000003730...",GTG/CTG,missense,V,L,"[{'predictionValType': 'benign', 'predictorTyp...",0,large_scale_study,NaN,"[{'type': 'Variant of uncertain significance',...",NaN,NaN,NaN
75,O43657,TSN6_HUMAN,Tetraspanin-6,TSPAN6,Homo sapiens,Evidence at protein level,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...,9304343482296458210,1,9606,VARIANT,T,87,87,"[{'name': '1000Genomes', 'id': 'rs145821935', ...",Xq22.1,[NC_000023.11:g.100635575C>T],"[{'loc': 'p.Ala87Thr', 'seqId': 'ENST000003730...",GCA/ACA,missense,A,T,"[{'predictionValType': 'benign', 'predictorTyp...",0,large_scale_study,NaN,"[{'type': 'Likely benign', 'sources': ['Ensemb...",NaN,NaN,NaN
109,O43657,TSN6_HUMAN,Tetraspanin-6,TSPAN6,Homo sapiens,Evidence at protein level,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...,9304343482296458210,1,9606,VARIANT,H,132,132,"[{'name': '1000Genomes', 'id': 'rs189159745', ...",Xq22.1,[NC_000023.11:g.100633985C>G],"[{'loc': 'p.Gln132His', 'seqId': 'ENST00000373...",CAG/CAC,missense,Q,H,"[{'predictionValType': 'benign', 'predictorTyp...",0,large_scale_study,NaN,"[{'type': 'Benign', 'sources': ['Ensembl']}]",NaN,NaN,NaN


### Compare gene metrics


In [ ]:
rgc = pd.read_excel("RGC-ME/41586_2024_7556_MOESM4_ESM.xlsx", sheet_name="table_s2")
rgc.head()

,GeneName,GeneId,TranscriptId,mean,sd,shet_lower,shet_upper,shet_constrained,n,N_total,mutation_rate,MAF,CDS_length,pKO,constraint_group,oe_lof,loeuf,loeuf_underpower,annotations
0,HSPB11,ENSG00000081870,ENST00000194214,0.038708,0.010539,0.023436,0.064243,False,15,1.643240e+06,3.410630e-07,0.000009,435,False,-,0.64467,1.347,0.0,lethal
1,RPL22,ENSG00000116251,ENST00000234875,0.132981,0.074968,0.050899,0.326520,True,4,1.642900e+06,2.946130e-07,0.000002,387,False,-,0.32990,1.036,0.0,"cancer,essential_line"
2,OPRD1,ENSG00000116329,ENST00000234961,0.008655,0.001007,0.006886,0.010856,False,76,1.643860e+06,3.951480e-07,0.000046,1119,True,LO,0.56836,1.193,0.0,ClinVar/HGMD
3,C1orf21,ENSG00000116667,ENST00000235307,0.120590,0.059463,0.051087,0.272217,True,5,1.643923e+06,3.419660e-07,0.000003,366,False,-,0.00000,0.414,0.0,NaN
4,RGS2,ENSG00000116741,ENST00000235382,0.003639,0.000245,0.003191,0.004155,False,222,1.643654e+06,4.884420e-07,0.000135,636,False,-,0.81481,1.417,0.0,"AR,ClinVar/HGMD"


In [ ]:
variant_counts = pd.DataFrame({k:len(set(v)) for k,v in variant_recorder.items()}, index=['variant_count']).T
variant_counts['TranscriptId'] = variant_counts.index.str.split('.').str[0]
variant_counts.head()
# merge on TranscriptId
variant_rgc = rgc.merge(variant_counts, on='TranscriptId', how='inner')
print(variant_rgc.shape)
variant_rgc.head()

(144, 20)


,GeneName,GeneId,TranscriptId,mean,sd,shet_lower,shet_upper,shet_constrained,n,N_total,mutation_rate,MAF,CDS_length,pKO,constraint_group,oe_lof,loeuf,loeuf_underpower,annotations,variant_count
0,BCL2L2,ENSG00000129473,ENST00000250405,0.011744,0.001896,0.008625,0.016013,False,40,1.643884e+06,2.803850e-07,0.000024,582,False,-,0.17414,0.826,0.0,NaN,16
1,DAD1,ENSG00000129562,ENST00000250498,0.024242,0.004953,0.016365,0.035704,False,25,1.643946e+06,3.601990e-07,0.000015,342,False,-,0.44592,1.369,1.0,"lethal,essential_line",4
2,SLC39A2,ENSG00000165794,ENST00000298681,0.000866,0.000029,0.000810,0.000924,False,868,1.643916e+06,4.548690e-07,0.000528,930,True,LO,0.66316,1.196,0.0,NaN,44
3,OR5AU1,ENSG00000169327,ENST00000304418,0.000101,0.000004,0.000093,0.000109,False,570,1.643951e+06,3.277900e-08,0.000347,936,True,LO,1.29240,1.891,0.0,NaN,59
4,RNASE3,ENSG00000169397,ENST00000304639,0.001827,0.000853,0.000820,0.003939,False,6,1.643952e+06,3.970000e-09,0.000004,483,False,LO,NaN,NaN,NaN,"HS,ClinVar/HGMD",23


In [ ]:
# Plot variant count vs RGC-ME  
import plotly.express as px
fig = px.scatter(variant_rgc, x='variant_count', y='n', hover_data=['TranscriptId'])
fig.show() 

## Extra variant objects

Take dbSNP/ClinVar variants and create a `pysam.VariantFile` object with them.

In [3]:
cv = get_clinvar_db()
get_clinvar_db_headers(cv)


['AF_ESP',
 'AF_EXAC',
 'AF_TGP',
 'ALLELEID',
 'CLNDN',
 'CLNDNINCL',
 'CLNDISDB',
 'CLNDISDBINCL',
 'CLNHGVS',
 'CLNREVSTAT',
 'CLNSIG',
 'CLNSIGCONF',
 'CLNSIGINCL',
 'CLNVC',
 'CLNVCSO',
 'CLNVI',
 'DBVARID',
 'GENEINFO',
 'MC',
 'ONCDN',
 'ONCDNINCL',
 'ONCDISDB',
 'ONCDISDBINCL',
 'ONC',
 'ONCINCL',
 'ONCREVSTAT',
 'ONCCONF',
 'ORIGIN',
 'RS',
 'SCIDN',
 'SCIDNINCL',
 'SCIDISDB',
 'SCIDISDBINCL',
 'SCIREVSTAT',
 'SCI',
 'SCIINCL']

In [11]:

recs, variant_counts = get_clinvar_variants(
    transcript_ids= mane['TranscriptId'].tolist()[:100],
    filters ={'CLNSIG': ('Pathogenic',
                        #  'Likely pathogenic',
                         ),
                'CLNREVSTAT': ('practice_guideline', # 4-star
                                # 'reviewed_by_expert_panel', # 3-star
                                )
                },
    coding_only=True,
    cv=cv,
    db=db,
    # save_path="mane_clinvar_variants.pkl",
    verbose=False
    )

Fetching clinvar variants:   0%|          | 0/1000 [00:00<?, ?it/s]

In [12]:
recs

In [14]:
selected_transcripts = {k:v for k,v in variant_counts.items() if v > 1}
print(len(selected_transcripts))
print(selected_transcripts)
# print([x for x ])
# filter_variants(recs, regions=["chr1:10000-100000"])


1
{'ENST00000003084': 16}


In [16]:
import os
os.chdir("/grid/koo/home/schilder/projects/GenomeEncoder/data")

# Load the Reference Genome
vcf_files = list_vcf()

results_all = personalize_seqs(vcf_files, 
                               ref_genome = "GRCh38/GRCh38_full_analysis_set_plus_decoy_hla.fa",
                               save_dir = "1KG/sequence_dict_all_extra_variants",
                               max_files = 1,
                               max_samples = 10,
                               max_transcripts = 10,
                            #    extra_variants = recs, # CLINVAR variants
                               transcript_ids = list(selected_transcripts.keys()),
                               max_workers = 1,
                               force = True
                               )

 

Processing VCF files:   0%|          | 0/1 [00:00<?, ?it/s]

23 VCF files found.
No transcripts found for chr14
Converting all results to dict.


In [45]:
results_all

{}

In [84]:
filter_variants(recs, regions=["chr1:10000-100000"])

0 variants remain after filtering


[]

In [80]:
recs[0].start

32316462